# Phase 3 Modeling

Phase 3 develops two classification systems for hospital decision support:

- **Model A: Visit Risk Classification** predicts `risk_score` as Low, Medium, or High.
- **Model B: Claim Outcome Classification** predicts `claim_status` as Paid, Pending, or Rejected.

The business goal is to support operational triage and proactive revenue-risk control with leakage-safe, reproducible model artifacts.

## Modeling Design

Both models use a time-based split by `visit_date`: the earliest 80% of records are used for training and the latest 20% are held out for testing. This mirrors production use, where models trained on historical visits are evaluated on future visits.

Leakage policy:

- Risk model excludes `claim_status`, approval fields, payment-delay fields, approval ratio, and claim-derived quality flags.
- Claim model excludes `approved_amount`, `payment_days`, `approval_ratio`, `provider_rejection_rate`, and any direct outcome-derived fields.
- Patient and visit identifiers are excluded from model features except `doctor_id`, which is treated as an operational categorical/numeric signal.

Baseline model: Logistic Regression with balanced class weights.

Advanced model: Random Forest with balanced subsampling.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_TABLE_PATH = ROOT / "data_outputs" / "model_table.csv"
ARTIFACT_DIR = ROOT / "model_artifacts"

df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["visit_date", "billing_date", "registration_date"])
df.shape

(25000, 44)

## Target Definitions

Model A target: `risk_score`

This target represents operational and clinical visit risk. It is useful for triage, resource planning, and identifying visits that may need additional review.

Model B target: `claim_status`

This target represents insurance claim outcome before final resolution. It supports billing workflow prioritization and revenue-risk monitoring.

In [2]:
target_summary = {
    "risk_score": df["risk_score"].value_counts().to_dict(),
    "claim_status": df["claim_status"].value_counts().to_dict(),
}
target_summary

{'risk_score': {'Low': 12470, 'Medium': 7496, 'High': 5034},
 'claim_status': {'Paid': 14940, 'Pending': 6263, 'Rejected': 3797}}

## Feature Sets

### Risk Model Features

The risk model uses patient demographics, visit context, length-of-stay signal, patient utilization, registration age, and time features:

```text
['age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'department', 'visit_type', 'doctor_id', 'length_of_stay_hours_filled', 'visit_frequency', 'days_since_registration', 'visit_month', 'visit_day_of_week', 'visit_quarter']
```

These features are business-relevant for clinical and operational triage while avoiding claim outcome leakage.

### Claim Model Features

The claim model uses demographics, payer, department, visit context, known clinical risk, billed amount, operational timing, and visit calendar features:

```text
['age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'department', 'visit_type', 'risk_score', 'doctor_id', 'billed_amount', 'length_of_stay_hours_filled', 'visit_frequency', 'days_since_registration', 'visit_month', 'visit_day_of_week', 'visit_quarter', 'billing_month', 'billing_lag_days']
```

Financial predictors are limited to values available before or around claim submission. Final approval and payment behavior are excluded.

In [3]:
from scripts.train_phase3_models import (
    CLAIM_FEATURES,
    CLAIM_TARGET,
    RISK_FEATURES,
    RISK_TARGET,
    time_split,
)

train_df, test_df = time_split(df)
split_summary = {
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "train_date_range": (train_df["visit_date"].min().date(), train_df["visit_date"].max().date()),
    "test_date_range": (test_df["visit_date"].min().date(), test_df["visit_date"].max().date()),
}
split_summary

{'train_rows': 20000,
 'test_rows': 5000,
 'train_date_range': (datetime.date(2025, 1, 20), datetime.date(2025, 11, 8)),
 'test_date_range': (datetime.date(2025, 11, 8), datetime.date(2026, 1, 20))}

## Class Balance

Risk target distribution:

|        |   train_share |   test_share |
|:-------|--------------:|-------------:|
| Low    |        0.4995 |       0.496  |
| Medium |        0.3    |       0.2994 |
| High   |        0.2006 |       0.2046 |

Claim target distribution:

|          |   train_share |   test_share |
|:---------|--------------:|-------------:|
| Paid     |        0.5972 |       0.5994 |
| Pending  |        0.2494 |       0.255  |
| Rejected |        0.1534 |       0.1456 |

The claim model has meaningful class imbalance: Paid claims are the majority class, while Rejected claims are the minority. The mitigation strategy in this phase is to use balanced class weights for Logistic Regression and balanced subsampling in Random Forest. Macro F1 is used as the main comparison metric because it gives each class equal importance.

![Phase 3 class balance](../reports/figures/phase3_class_balance.png)

## Train Baseline and Advanced Models

Running this cell retrains both model families and writes deployable artifacts to `model_artifacts/`.

In [4]:
from scripts.train_phase3_models import main as train_phase3_models

train_phase3_models()

    problem               model  accuracy  macro_f1  weighted_f1                                         artifact
 risk_model logistic_regression    0.3142    0.3058       0.3204  model_artifacts/risk_logistic_regression.joblib
 risk_model       random_forest    0.4102    0.3491       0.4018        model_artifacts/risk_random_forest.joblib
claim_model logistic_regression    0.3526    0.3231       0.3801 model_artifacts/claim_logistic_regression.joblib
claim_model       random_forest    0.4442    0.3753       0.4533       model_artifacts/claim_random_forest.joblib
wrote artifacts to: /home/suuri/Downloads/capstone-iitm/model_artifacts


In [5]:
metrics = json.loads((ARTIFACT_DIR / "metrics.json").read_text())
metrics_summary = pd.read_csv(ARTIFACT_DIR / "metrics_summary.csv")
metrics_summary

,problem,model,accuracy,macro_f1,weighted_f1,artifact
0,risk_model,logistic_regression,0.3142,0.3058,0.3204,model_artifacts/risk_logistic_regression.joblib
1,risk_model,random_forest,0.4102,0.3491,0.4018,model_artifacts/risk_random_forest.joblib
2,claim_model,logistic_regression,0.3526,0.3231,0.3801,model_artifacts/claim_logistic_regression.joblib
3,claim_model,random_forest,0.4442,0.3753,0.4533,model_artifacts/claim_random_forest.joblib


## Evaluation Results

| problem     | model               |   accuracy |   macro_f1 |   weighted_f1 | artifact                                         |
|:------------|:--------------------|-----------:|-----------:|--------------:|:-------------------------------------------------|
| risk_model  | logistic_regression |     0.3142 |     0.3058 |        0.3204 | model_artifacts/risk_logistic_regression.joblib  |
| risk_model  | random_forest       |     0.4092 |     0.3485 |        0.4013 | model_artifacts/risk_random_forest.joblib        |
| claim_model | logistic_regression |     0.3526 |     0.3231 |        0.3801 | model_artifacts/claim_logistic_regression.joblib |
| claim_model | random_forest       |     0.4444 |     0.3743 |        0.4525 | model_artifacts/claim_random_forest.joblib       |

Interpretation:

- Random Forest is the stronger model for both tasks by macro F1.
- Logistic Regression remains useful as a transparent baseline, but it underfits the nonlinear relationships in mixed hospital operations and billing data.
- Absolute scores are modest, which is a useful business finding: the current structured fields provide a baseline signal but likely need richer diagnosis, procedure, doctor workload, authorization, and claim-history features for production-grade performance.

![Phase 3 model comparison](../reports/figures/phase3_model_comparison.png)

## Confusion Matrix Review

The confusion matrices below use the Random Forest models because they are the best-performing models in this phase. They show where the model confuses neighboring operational risk levels and claim outcomes.

![Phase 3 confusion matrices](../reports/figures/phase3_confusion_matrices.png)

In [6]:
for problem_name in ["risk_model", "claim_model"]:
    best_model = metrics[problem_name]["best_model"]
    report = metrics[problem_name]["models"][best_model]["metrics"]["classification_report"]
    print(problem_name, "best model:", best_model)
    display(pd.DataFrame(report).transpose().round(4))

risk_model best model: random_forest


,precision,recall,f1-score,support
High,0.2194,0.1261,0.1601,1023.0000
Low,0.5287,0.5238,0.5262,2480.0000
Medium,0.3187,0.4162,0.3610,1497.0000
accuracy,0.4102,0.4102,0.4102,0.4102
macro avg,0.3556,0.3554,0.3491,5000.0000
weighted avg,0.4025,0.4102,0.4018,5000.0000


claim_model best model: random_forest


,precision,recall,f1-score,support
Paid,0.6636,0.5265,0.5872,2997.0000
Pending,0.2966,0.1624,0.2098,1275.0000
Rejected,0.2266,0.5989,0.3288,728.0000
accuracy,0.4442,0.4442,0.4442,0.4442
macro avg,0.3956,0.4293,0.3753,5000.0000
weighted avg,0.5064,0.4442,0.4533,5000.0000


## Saved Artifacts

The training workflow saves both baseline and advanced models, plus a best-model pointer for each target.

Risk model artifacts:

- `model_artifacts/risk_logistic_regression.joblib`
- `model_artifacts/risk_random_forest.joblib`
- `model_artifacts/risk_best_model.joblib`

Claim model artifacts:

- `model_artifacts/claim_logistic_regression.joblib`
- `model_artifacts/claim_random_forest.joblib`
- `model_artifacts/claim_best_model.joblib`

Supporting files:

- `model_artifacts/feature_schema.json`
- `model_artifacts/metrics.json`
- `model_artifacts/metrics_summary.csv`

In [7]:
schema = json.loads((ARTIFACT_DIR / "feature_schema.json").read_text())
schema

{'phase': 'Phase 3 Modeling',
 'source_dataset': 'data_outputs/model_table.csv',
 'split_strategy': 'time-based split by visit_date: earliest 80% train, latest 20% test',
 'leakage_policy': {'risk_model': 'Excludes claim_status, approval, payment delay, approval ratio, and claim-derived fields.',
  'claim_model': 'Excludes approved_amount, payment_days, approval_ratio, provider_rejection_rate, and target-derived outcome fields.'},
 'models': {'risk_model': {'target': 'risk_score',
   'features': ['age',
    'gender',
    'city',
    'insurance_provider',
    'chronic_flag',
    'department',
    'visit_type',
    'doctor_id',
    'length_of_stay_hours_filled',
    'visit_frequency',
    'days_since_registration',
    'visit_month',
    'visit_day_of_week',
    'visit_quarter'],
   'categorical_features': ['gender',
    'city',
    'insurance_provider',
    'department',
    'visit_type'],
   'best_model': 'random_forest',
   'best_artifact': 'model_artifacts/risk_best_model.joblib'},
 

In [8]:
risk_model = joblib.load(ARTIFACT_DIR / "risk_best_model.joblib")
claim_model = joblib.load(ARTIFACT_DIR / "claim_best_model.joblib")

sample = test_df.iloc[[0]]
risk_prediction = risk_model.predict(sample[RISK_FEATURES])[0]
claim_prediction = claim_model.predict(sample[CLAIM_FEATURES])[0]
{
    "visit_id": int(sample["visit_id"].iloc[0]),
    "actual_risk_score": sample[RISK_TARGET].iloc[0],
    "predicted_risk_score": risk_prediction,
    "actual_claim_status": sample[CLAIM_TARGET].iloc[0],
    "predicted_claim_status": claim_prediction,
}

{'visit_id': 9322,
 'actual_risk_score': 'Low',
 'predicted_risk_score': 'Low',
 'actual_claim_status': 'Paid',
 'predicted_claim_status': 'Paid'}

## Business Conclusions

- The advanced Random Forest models are the deployable Phase 3 candidates because they outperform the Logistic Regression baselines on macro F1.
- The risk model can support first-pass visit triage, but current features should be treated as baseline operational signals rather than full clinical decision support.
- The claim model provides a practical starting point for revenue-risk prioritization, especially because it avoids using final approval or payment fields.
- Class imbalance is material for claim outcomes; future iterations should evaluate thresholding, calibrated probabilities, and additional payer-history features.
- The saved feature schema creates a consistent contract between training, API scoring, and future Phase 4 dashboard or decision-support workflows.